In [1]:
import pandas as pd
import numpy as np
import os

# =====================================================================
# 1. INDEPENDENT HEALTHCARE DATA ENGINE (Generates data if missing)
# =====================================================================
FILE_NAME = "patient_records.csv"

if not os.path.exists(FILE_NAME):
    print(f"🔍 '{FILE_NAME}' not found. Initializing a brand-new medical dataset...")
    np.random.seed(18)
    total_records = 250
    
    # Generate realistic messy clinical records
    raw_data = {
        "Patient_ID": [np.nan if i in [12, 45, 112] else int(2000 + i) for i in range(total_records)],
        "Age": [np.nan if i in [5, 78, 142, 210] else int(np.random.randint(12, 90)) for i in range(total_records)],
        "Gender": np.random.choice(["Male", "Female", "M", "F", "female", "male"], size=total_records),
        "Clinical_Diagnosis": np.random.choice(["Hypertension", "Type 2 Diabetes", "Asthma", "Chronic Kidney Disease", "Arthritis"], size=total_records),
        "Prescribed_Drug": np.random.choice(["Lisinopril", "Metformin", "Albuterol", "Losartan", "Ibuprofen", np.nan], size=total_records),
        "Classification": np.random.choice(["Branded", "Generic"], size=total_records),
        "Revenue_INR": [np.nan if i in [30, 88] else round(float(np.random.uniform(250.0, 4500.0)), 2) for i in range(total_records)],
        "Reported_ADR": np.random.choice(["None", "Mild Rash", "Headache", "Nausea", "Dizziness", "Fatigue"], size=total_records, p=[0.6, 0.08, 0.08, 0.08, 0.08, 0.08]),
        "High_Risk_Flag": np.random.choice([1, 0], size=total_records)
    }
    
    base_df = pd.DataFrame(raw_data)
    
    # Inject 12 structural row duplicates deliberately
    duplicate_subset = base_df.iloc[20:32]
    final_raw_df = pd.concat([base_df, duplicate_subset], ignore_index=True)
    final_raw_df.to_csv(FILE_NAME, index=False)
    print(f"📦 Successfully generated '{FILE_NAME}' ({len(final_raw_df)} rows) with structured anomalies.\n")

# =====================================================================
# 2. PIPELINE: DATA CLEANING & STATISTICAL UNDERSTANDING
# =====================================================================
print("🏁 Executing Data Cleaning Pipeline...")
df = pd.read_csv(FILE_NAME)

# A. Remove exact duplicate records
initial_row_count = len(df)
df = df.drop_duplicates()
print(f"🧹 Removed {initial_row_count - len(df)} duplicate records.")

# B. Drop rows where key medical identifiers (Patient_ID) are missing
id_missing_count = df["Patient_ID"].isnull().sum()
df = df.dropna(subset=["Patient_ID"])
df["Patient_ID"] = df["Patient_ID"].astype(int) # Standardize IDs to integers
print(f"🧹 Dropped {id_missing_count} rows missing a critical Patient_ID.")

# C. Fill missing clinical metadata using safe analytical metrics
# Fill missing age with the median age of the group
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age).astype(int)

# Fill missing medications with a placeholder
df["Prescribed_Drug"] = df["Prescribed_Drug"].fillna("No Medication Prescribed")

# Impute missing sales revenue with the calculated mean revenue value
mean_revenue = df["Revenue_INR"].mean()
df["Revenue_INR"] = df["Revenue_INR"].fillna(mean_revenue).round(2)

print("📝 Successfully completed missing data imputation strategies.")

# D. Standardize clinical categorical entry names
gender_clean_map = {
    'M': 'Male', 'male': 'Male', 'Male': 'Male',
    'F': 'Female', 'female': 'Female', 'Female': 'Female'
}
df["Gender"] = df["Gender"].map(gender_clean_map).fillna("Unknown")
print("🔤 Uniform categorical mapping applied to structural strings.")

# =====================================================================
# 3. ANALYSIS SUMMARIES & PIPELINE OUTPUT EXPORT
# =====================================================================
print(f"\n✅ Processing complete. Total valid records in clean pipeline: {len(df)}")

print("\n📊 --- NUMERICAL METRICS SUMMARY ---")
display(df.describe())

print("\n📁 --- CATEGORICAL VALUE COUNTS ---")
display(df["Clinical_Diagnosis"].value_counts().to_frame())

# Export the clean file for Task 2 (Power BI visuals) and Task 3 (SQL)
CLEAN_FILE_NAME = "cleaned_patient_records.csv"
df.to_csv(CLEAN_FILE_NAME, index=False)
print(f"\n💾 Pipeline output exported successfully as '{CLEAN_FILE_NAME}'!")

ModuleNotFoundError: No module named 'pandas'